CONSTITUENTS_CURRENT

In [ ]:
import pandas as pd
url = "https://en.wikipedia.org/wiki/NIFTY_50"

# Read all tables from the Wikipedia page
tables = pd.read_html(url)

# Find the table with all 4 required columns
target_df = None
for table in tables:
    if {"Company name", "Symbol", "Sector[15]", "Date added[16]"}.issubset(table.columns):
        target_df = table
        break

# If table not found
if target_df is None:
    raise ValueError("NIFTY 50 constituents table not found. Page structure may have changed.")

# Rename columns
target_df = target_df.rename(columns={
    "Company name": "Company Name",
    "Symbol": "Symbol",
    "Sector[15]": "Sector",
    "Date added[16]": "Date Added"
})

# Reorder and keep only needed columns
nifty_df = target_df[["Company Name", "Symbol", "Sector", "Date Added"]]
print(nifty_df)


                       Company Name      Symbol  \
0                 Adani Enterprises    ADANIENT   
1                 Adani Ports & SEZ  ADANIPORTS   
2                  Apollo Hospitals  APOLLOHOSP   
3                      Asian Paints  ASIANPAINT   
4                         Axis Bank    AXISBANK   
5                        Bajaj Auto  BAJAJ-AUTO   
6                     Bajaj Finance  BAJFINANCE   
7                     Bajaj Finserv  BAJAJFINSV   
8                Bharat Electronics         BEL   
9                     Bharti Airtel  BHARTIARTL   
10                            Cipla       CIPLA   
11                       Coal India   COALINDIA   
12         Dr. Reddy's Laboratories     DRREDDY   
13                    Eicher Motors   EICHERMOT   
14                          Eternal     ETERNAL   
15                Grasim Industries      GRASIM   
16                          HCLTech     HCLTECH   
17                        HDFC Bank    HDFCBANK   
18                        HDFC 

PAST_CONSTITUENTS

In [ ]:
import pandas as pd
url = "https://en.wikipedia.org/wiki/NIFTY_50"
tables = pd.read_html(url, header=[0, 1])  # MultiIndex
changes_df = tables[2]  # This is the "Index changes" table

# Step 2: Drop the last column (Ref), keep first 4
changes_df = changes_df.iloc[:, :4]

# Step 3: Rename columns manually
changes_df.columns = ["Excluded", "Included", "Date", "Reason"]
print(changes_df.head())


                        Excluded                 Included  \
0           Constituent excluded     Constituent included   
1  Shipping Corporation of India            Siemens India   
2                 Tata Chemicals                   Suzlon   
3                       Tata Tea  Reliance Communications   
4                    Jet Airways       Reliance Petroleum   

                  Date                            Reason  
0  Date of replacement              Reason for exclusion  
1         27 June 2006  Inadequate market capitalization  
2         27 June 2006  Inadequate market capitalization  
3     1 September 2006  Inadequate market capitalization  
4         4 April 2007  Inadequate market capitalization  


In [ ]:
current_nifty = nifty_df['Company Name'].tolist()

In [ ]:
changes_df = changes_df.drop(index=0).reset_index(drop=True)

In [ ]:
changes_df

,Excluded,Included,Date,Reason
0,Shipping Corporation of India,Siemens India,27 June 2006,Inadequate market capitalization
1,Tata Chemicals,Suzlon,27 June 2006,Inadequate market capitalization
2,Tata Tea,Reliance Communications,1 September 2006,Inadequate market capitalization
3,Jet Airways,Reliance Petroleum,4 April 2007,Inadequate market capitalization
4,Oriental Bank of Commerce,Sterlite Industries,4 April 2007,Inadequate market capitalization
...,...,...,...,...
60,UPL,Shriram Finance,28 March 2024,Inadequate market capitalization
61,Divi's Laboratories,Bharat Electronics,30 September 2024,Inadequate market capitalization
62,LTIMindtree,Trent,30 September 2024,Inadequate market capitalization
63,Bharat Petroleum,Jio Financial Services,28 March 2025,Inadequate market capitalization


In [ ]:
changes_df_rev = changes_df[::-1]
changes_df_rev

,Excluded,Included,Date,Reason
64,Britannia Industries,Eternal,28 March 2025,Inadequate market capitalization
63,Bharat Petroleum,Jio Financial Services,28 March 2025,Inadequate market capitalization
62,LTIMindtree,Trent,30 September 2024,Inadequate market capitalization
61,Divi's Laboratories,Bharat Electronics,30 September 2024,Inadequate market capitalization
60,UPL,Shriram Finance,28 March 2024,Inadequate market capitalization
...,...,...,...,...
4,Oriental Bank of Commerce,Sterlite Industries,4 April 2007,Inadequate market capitalization
3,Jet Airways,Reliance Petroleum,4 April 2007,Inadequate market capitalization
2,Tata Tea,Reliance Communications,1 September 2006,Inadequate market capitalization
1,Tata Chemicals,Suzlon,27 June 2006,Inadequate market capitalization


In [ ]:
current_nifty_cp = current_nifty.copy()

In [ ]:
## grouping with same dates
import pandas as pd

grouped_df = changes_df_rev.groupby('Date', sort = False).agg({
    'Excluded': list,
    'Included': list
}).reset_index()
print(grouped_df.head())


                Date                                  Excluded  \
0      28 March 2025  [Britannia Industries, Bharat Petroleum]   
1  30 September 2024        [LTIMindtree, Divi's Laboratories]   
2      28 March 2024                                     [UPL]   
3       13 July 2023                                    [HDFC]   
4  30 September 2022                            [Shree Cement]   

                            Included  
0  [Eternal, Jio Financial Services]  
1        [Trent, Bharat Electronics]  
2                  [Shriram Finance]  
3                      [LTIMindtree]  
4                [Adani Enterprises]  


In [ ]:
current_nifty_cp = current_nifty.copy()
yearwise_dict = {}
# for i in range(changes_df_rev.shape[0]):
for i in range(grouped_df.shape[0]):
  index = current_nifty_cp.copy()
  # print(index)
  date = grouped_df.iloc[i]['Date']
  included = grouped_df.iloc[i]['Included']
  excluded = grouped_df.iloc[i]['Excluded']
  print(date, included, excluded)
  for i in included:
    if i not in index:
      index.append(i)
  for i in excluded:
    if i in index:
      index.remove(i)
  # print(index)

  yearwise_dict[date] = index
  index_changed = index.copy()
  # ## now to do back reconstruction, remove included and add excluded
  for i in included:
    index_changed.remove(i)
  for i in excluded:
    index_changed.append(i)
  # print(index_changed)

  current_nifty_cp = index_changed

28 March 2025 ['Eternal', 'Jio Financial Services'] ['Britannia Industries', 'Bharat Petroleum']
30 September 2024 ['Trent', 'Bharat Electronics'] ['LTIMindtree', "Divi's Laboratories"]
28 March 2024 ['Shriram Finance'] ['UPL']
13 July 2023 ['LTIMindtree'] ['HDFC']
30 September 2022 ['Adani Enterprises'] ['Shree Cement']
31 March 2022 ['Apollo Hospitals'] ['Indian Oil Corporation']
31 March 2021 ['Tata Consumer Products'] ['GAIL']
25 September 2020 ["Divi's Laboratories", 'SBI Life Insurance Company'] ['Bharti Infratel', 'Zee Entertainment Enterprises']
31 July 2020 ['HDFC Life'] ['Vedanta']
19 March 2020 ['Shree Cement'] ['Yes Bank']
27 September 2019 ['Nestlé India'] ['Indiabulls Housing Finance']
29 March 2019 ['Britannia Industries'] ['Hindustan Petroleum']
28 September 2018 ['JSW Steel'] ['Lupin']
2 April 2018 ['Titan Company', 'Grasim Industries', 'Bajaj Finserv'] ['Bosch India', 'Aurobindo Pharma', 'Ambuja Cements']
29 September 2017 ['UPL', 'Hindustan Petroleum', 'Bajaj Finance

In [ ]:
for i in yearwise_dict.values():
  print(len(i))

50
50
50
50
50
50
50
50
50
50
50
50
50
50
50
50
50
50
50
50
50
50
50
50
50
50
50
50
50
51
51
52
52
52
52
52
52
52
52
52
52
52
52


In [ ]:
final_df = pd.DataFrame.from_dict(yearwise_dict, orient='index')
final_df.reset_index(inplace=True)
final_df = final_df.rename(columns={'index': 'Date'})

In [ ]:
final_df

,Date,0,1,2,3,4,5,6,7,8,...,42,43,44,45,46,47,48,49,50,51
0,28 March 2025,Adani Enterprises,Adani Ports & SEZ,Apollo Hospitals,Asian Paints,Axis Bank,Bajaj Auto,Bajaj Finance,Bajaj Finserv,Bharat Electronics,...,Tata Consumer Products,Tata Motors,Tata Steel,Tech Mahindra,Titan Company,Trent,UltraTech Cement,Wipro,None,None
1,30 September 2024,Adani Enterprises,Adani Ports & SEZ,Apollo Hospitals,Asian Paints,Axis Bank,Bajaj Auto,Bajaj Finance,Bajaj Finserv,Bharat Electronics,...,Tata Steel,Tech Mahindra,Titan Company,Trent,UltraTech Cement,Wipro,Britannia Industries,Bharat Petroleum,None,None
2,28 March 2024,Adani Enterprises,Adani Ports & SEZ,Apollo Hospitals,Asian Paints,Axis Bank,Bajaj Auto,Bajaj Finance,Bajaj Finserv,Bharti Airtel,...,Tech Mahindra,Titan Company,UltraTech Cement,Wipro,Britannia Industries,Bharat Petroleum,LTIMindtree,Divi's Laboratories,None,None
3,13 July 2023,Adani Enterprises,Adani Ports & SEZ,Apollo Hospitals,Asian Paints,Axis Bank,Bajaj Auto,Bajaj Finance,Bajaj Finserv,Bharti Airtel,...,Titan Company,UltraTech Cement,Wipro,Britannia Industries,Bharat Petroleum,LTIMindtree,Divi's Laboratories,UPL,None,None
4,30 September 2022,Adani Enterprises,Adani Ports & SEZ,Apollo Hospitals,Asian Paints,Axis Bank,Bajaj Auto,Bajaj Finance,Bajaj Finserv,Bharti Airtel,...,Titan Company,UltraTech Cement,Wipro,Britannia Industries,Bharat Petroleum,Divi's Laboratories,UPL,HDFC,None,None
5,31 March 2022,Adani Ports & SEZ,Apollo Hospitals,Asian Paints,Axis Bank,Bajaj Auto,Bajaj Finance,Bajaj Finserv,Bharti Airtel,Cipla,...,UltraTech Cement,Wipro,Britannia Industries,Bharat Petroleum,Divi's Laboratories,UPL,HDFC,Shree Cement,None,None
6,31 March 2021,Adani Ports & SEZ,Asian Paints,Axis Bank,Bajaj Auto,Bajaj Finance,Bajaj Finserv,Bharti Airtel,Cipla,Coal India,...,Wipro,Britannia Industries,Bharat Petroleum,Divi's Laboratories,UPL,HDFC,Shree Cement,Indian Oil Corporation,None,None
7,25 September 2020,Adani Ports & SEZ,Asian Paints,Axis Bank,Bajaj Auto,Bajaj Finance,Bajaj Finserv,Bharti Airtel,Cipla,Coal India,...,Britannia Industries,Bharat Petroleum,Divi's Laboratories,UPL,HDFC,Shree Cement,Indian Oil Corporation,GAIL,None,None
8,31 July 2020,Adani Ports & SEZ,Asian Paints,Axis Bank,Bajaj Auto,Bajaj Finance,Bajaj Finserv,Bharti Airtel,Cipla,Coal India,...,Bharat Petroleum,UPL,HDFC,Shree Cement,Indian Oil Corporation,GAIL,Bharti Infratel,Zee Entertainment Enterprises,None,None
9,19 March 2020,Adani Ports & SEZ,Asian Paints,Axis Bank,Bajaj Auto,Bajaj Finance,Bajaj Finserv,Bharti Airtel,Cipla,Coal India,...,UPL,HDFC,Shree Cement,Indian Oil Corporation,GAIL,Bharti Infratel,Zee Entertainment Enterprises,Vedanta,None,None


In [ ]:
final_df.to_csv('yearwise_niftyfinall.csv', index=False)